# oWAR Overview Dashboard - Current Season Projections

**Purpose:** All-in-one dashboard for 2025 pitcher and hitter WAR projections

**Last Updated:** 2025-10-13

---

## Features
- Interactive scatter plots (WAR vs IP/PA)
- Featured player tables with rankings
- Two-way player support (Shohei Ohtani)
- Rest of season (ROS) projections

## Recent Updates (2025-10-13)
- Fixed IP convergence issue - pitchers now project naturally without artificial clustering
- Removed 210 IP cap for starters - allows natural workload projections
- Improved swing pitcher projections - 140 IP cap instead of 100 IP
- Note: ROS projections may be conservative - calibration under review

In [1]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd

# Add project root to path
project_root = Path('.').absolute().parent.parent
sys.path.insert(0, str(project_root))

from new_pipeline.notebooks.shared.pipeline_runner import (
    load_current_season_data,
    run_data_pipeline,
    generate_predictions
)
from new_pipeline.notebooks.shared.plotting_utils import create_war_scatter
from new_pipeline.notebooks.shared.table_utils import (
    create_featured_table,
    handle_two_way_player
)
from new_pipeline.models.current_season import PitcherRoleEnsemble, HitterEnsemble

# Register Keras custom loss function for model loading
import keras
from new_pipeline.models.current_season.keras_utils import multi_quantile_loss

# Create the specific loss function instance used in saved models
# Parameters MUST match those used during training (see keras_utils.py line 174-177)
loss_fn = multi_quantile_loss([0.5, 0.75, 0.9], weights=[0.2, 0.3, 0.5])

# Manually register with name Keras expects during deserialization
keras.saving.get_custom_objects()['multi_quantile_loss_[0.5, 0.75, 0.9]'] = loss_fn

print("Imports successful!")
print("Custom loss function registered with Keras")

17:30:26 - new_pipeline.common.logging_config - INFO - Logging module initialized for new_pipeline
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful!
Custom loss function registered with Keras


In [2]:
# Cell 2: Load Current Season Data and Models

print("Loading 2025 current season data...")

# Load raw data
pitcher_raw = load_current_season_data('pitcher', year=2025)
hitter_raw = load_current_season_data('hitter', year=2025)

print(f"Loaded {len(pitcher_raw)} pitchers (raw)")
print(f"Loaded {len(hitter_raw)} hitters (raw)")

# Clean multi-team players BEFORE processing
from new_pipeline.common.data_preparation.clean_multi_team_data import clean_multi_team_players

print("\nCleaning multi-team players...")
pitcher_raw = clean_multi_team_players(pitcher_raw, year=2025, player_type='pitcher')
hitter_raw = clean_multi_team_players(hitter_raw, year=2025, player_type='hitter')

print(f"After multi-team cleaning: {len(pitcher_raw)} pitchers, {len(hitter_raw)} hitters")

# Run pipelines
print("\nRunning data pipelines...")
pitcher_processed = run_data_pipeline(pitcher_raw, player_type='pitcher')
hitter_processed = run_data_pipeline(hitter_raw, player_type='hitter')

print(f"Processed {len(pitcher_processed)} pitchers (qualified)")
print(f"Processed {len(hitter_processed)} hitters (qualified)")

# Load current season models using proper methods
models_dir = project_root / 'models'

print("\n" + "="*70)
print("LOADING CURRENT SEASON MODELS")
print("="*70)

# Load pitcher model
pitcher_model_base = str(models_dir / 'pitcher_role_ensemble_2025')
pitcher_meta_path = models_dir / 'pitcher_role_ensemble_2025_meta.pkl'

if pitcher_meta_path.exists():
    print("\nLoading pre-trained pitcher model...")
    pitcher_model = PitcherRoleEnsemble()
    pitcher_model.load(pitcher_model_base)
    print(f"  Loaded: pitcher_role_ensemble_2025 (7 files)")
    print("  Model trained on historical data (2016-2024)")
else:
    print("\nERROR: Pitcher model not found!")
    print(f"  Expected: {pitcher_meta_path}")
    print("\nPlease run pitcher_pipeline_main.ipynb first to train model on historical data.")
    print("  Location: new_pipeline/notebooks/pitchers/pitcher_pipeline_main.ipynb")
    raise FileNotFoundError(f"Pitcher model not found: {pitcher_meta_path}")

# Load hitter model
hitter_model_base = str(models_dir / 'hitter_ensemble_2025')
hitter_meta_path = models_dir / 'hitter_ensemble_2025.pkl'

if hitter_meta_path.exists():
    print("\nLoading pre-trained hitter model...")
    hitter_model = HitterEnsemble()
    hitter_model.load(hitter_model_base)
    print(f"  Loaded: hitter_ensemble_2025")
    print("  Model trained on historical data (2016-2024)")
else:
    print("\nERROR: Hitter model not found!")
    print(f"  Expected: {hitter_meta_path}")
    print("\nPlease run hitter_pipeline_main.ipynb first to train model on historical data.")
    print("  Location: new_pipeline/notebooks/hitters/hitter_pipeline_main.ipynb")
    raise FileNotFoundError(f"Hitter model not found: {hitter_meta_path}")

print("\n" + "="*70)
print("Current season models loaded successfully!")
print("Ready for ROS model loading and complete projections...")
print("="*70)

17:30:36 - new_pipeline.common.transformers.filters - INFO - IPFilter: Removed 157 pitchers (position players / insufficient sample, partial season)
17:30:36 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 873 pitchers
17:30:36 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 20-42)
17:30:36 - new_pipeline.common.transformers.pitcher_features - INFO - Loading pitcher features...


Loading 2025 current season data...
Loading partial season data: fangraphs_pitchers_2025_firsthalf.csv
Loading partial season data: fangraphs_hitters_2025_firsthalf.csv
Loaded 754 pitchers (raw)
Loaded 606 hitters (raw)

Cleaning multi-team players...
Found 33 multi-team pitcher(s) to clean
[STINT CACHE DEBUG] Year=2025, Cache path=c:\Users\nairs\Documents\GithubProjects\oWAR\cache\team_stints_2025.json
[STINT CACHE DEBUG] Loaded cache with 58 existing entries
Using cached stint data for player 16943 (2025)
  Sean Newcomb: ATH, BOS (current: BOS)
[STINT CACHE DEBUG] Year=2025, Cache path=c:\Users\nairs\Documents\GithubProjects\oWAR\cache\team_stints_2025.json
[STINT CACHE DEBUG] Loaded cache with 58 existing entries
Using cached stint data for player 17735 (2025)
  Tyler Alexander: CHW, MIL (current: MIL)
[STINT CACHE DEBUG] Year=2025, Cache path=c:\Users\nairs\Documents\GithubProjects\oWAR\cache\team_stints_2025.json
[STINT CACHE DEBUG] Loaded cache with 58 existing entries
Using cach

17:30:36 - new_pipeline.common.transformers.pitcher_features - INFO - Loaded 13 pitcher feature sets (40 total columns)
17:30:36 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Calculating pitcher composite features...
17:30:36 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Added 7 composite features
17:30:36 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 41 features
17:30:36 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 883 missing values
17:30:36 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'BB%' range [0.00, 26.92] outside expected [0, 25]
  - Feature 'ERA' range [0.00, 19.86] outside expected [0, 15]
  - Feature 'GB%' range [11.11, 74.71] outside expected [20, 80]
17:30:36 - new_pipeline.common.transformers.feature_selector - INFO - FeatureSelector: Selected 14 features + 12 met

17:30:36 - new_pipeline.common.transformers.normalizers - INFO - WARNormalizer: Added 'WAR_per_162' column (role-specific: 198 starters/162IP, 338 relievers/48.2IP, 61 swing/110IP)
17:30:36 - new_pipeline.common.transformers.filters - INFO - PAFilter: Removed 118 hitters with < 37 PA (partial season)
17:30:36 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 673 hitters
17:30:36 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 21-41)
17:30:36 - new_pipeline.common.transformers.hitter_features - INFO - Loading hitter features...
17:30:37 - new_pipeline.common.transformers.hitter_features - INFO - Loaded 11 hitter feature sets (35 total columns)
17:30:37 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 28 features
17:30:37 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 99 missing values
17:30:37 - new_pipeline.comm

Processed 597 pitchers (qualified)
Processed 488 hitters (qualified)

LOADING CURRENT SEASON MODELS

Loading pre-trained pitcher model...
  Loaded: pitcher_role_ensemble_2025 (7 files)
  Model trained on historical data (2016-2024)

Loading pre-trained hitter model...
  Loaded: hitter_ensemble_2025
  Model trained on historical data (2016-2024)

Current season models loaded successfully!
Ready for ROS model loading and complete projections...


In [3]:
# Cell 3: Load ROS Models and Historical Data

print("="*70)
print("LOADING ROS MODELS FOR COMPLETE PROJECTIONS")
print("="*70)

import joblib
from darts.models import TCNModel, TSMixerModel
from new_pipeline.common.projections import CompleteProjectionGenerator

# Check if trained ROS models exist
models_dir = project_root / 'models'
hitter_ros_path = models_dir / 'hitter_ros_2025.pkl'
pitcher_ros_path = models_dir / 'pitcher_ros_2025.pkl'

if hitter_ros_path.exists() and pitcher_ros_path.exists():
    print("\nLoading trained ROS models...")
    hitter_ros = joblib.load(hitter_ros_path)
    pitcher_ros = joblib.load(pitcher_ros_path)
    print("  Hitter ROS model loaded")
    print("  Pitcher ROS model loaded")

    print("\nLoading Darts temporal models...")
    
    # Hitter temporal models
    hitter_tcn_path = models_dir / 'hitter_ros_tcn_2025.pt'
    hitter_tsmixer_path = models_dir / 'hitter_ros_tsmixer_2025.pt'
    
    if hitter_tcn_path.exists() and hitter_tsmixer_path.exists():
        hitter_ros.temporal_model.tcn = TCNModel.load(str(hitter_tcn_path))
        hitter_ros.temporal_model.tsmixer = TSMixerModel.load(str(hitter_tsmixer_path))
        print("  Hitter TCN and TSMixer loaded")
        
        # Verify .model attribute is not None
        assert hitter_ros.temporal_model.tcn.model is not None, "Hitter TCN .model is None!"
        assert hitter_ros.temporal_model.tsmixer.model is not None, "Hitter TSMixer .model is None!"
        print("  Verified: Hitter temporal models have .model attribute")
    else:
        print("  Warning: Hitter Darts models not found, temporal predictions may fail")
        print(f"    Expected: {hitter_tcn_path}")
        print(f"    Expected: {hitter_tsmixer_path}")
    
    # Pitcher temporal models
    pitcher_tcn_path = models_dir / 'pitcher_ros_tcn_2025.pt'
    pitcher_tsmixer_path = models_dir / 'pitcher_ros_tsmixer_2025.pt'
    
    if pitcher_tcn_path.exists() and pitcher_tsmixer_path.exists():
        pitcher_ros.temporal_model.tcn = TCNModel.load(str(pitcher_tcn_path))
        pitcher_ros.temporal_model.tsmixer = TSMixerModel.load(str(pitcher_tsmixer_path))
        print("  Pitcher TCN and TSMixer loaded")
        
        # Verify .model attribute is not None
        assert pitcher_ros.temporal_model.tcn.model is not None, "Pitcher TCN .model is None!"
        assert pitcher_ros.temporal_model.tsmixer.model is not None, "Pitcher TSMixer .model is None!"
        print("  Verified: Pitcher temporal models have .model attribute")
    else:
        print("  Warning: Pitcher Darts models not found, temporal predictions may fail")
        print(f"    Expected: {pitcher_tcn_path}")
        print(f"    Expected: {pitcher_tsmixer_path}")

    def fix_all_forecaster_cutoffs(ros_model, model_name):
        """
        Fix frequency for all forecaster cutoffs (main + individual).
        
        sktime creates individual forecasters during fit() that lose frequency
        metadata, even when input data has frequency. This patches them after loading.
        """
        if not hasattr(ros_model, 'direct_model'):
            print(f"  {model_name}: No direct_model, skipping")
            return
        
        forecaster = ros_model.direct_model.forecaster
        
        def fix_single_cutoff(cutoff_obj):
            """Rebuild a DatetimeIndex with frequency."""
            if cutoff_obj is None or not isinstance(cutoff_obj, pd.DatetimeIndex):
                return cutoff_obj
            
            if cutoff_obj.freq is not None:
                return cutoff_obj  # Already has frequency
            
            # Rebuild DatetimeIndex with frequency parameter
            try:
                new_cutoff = pd.DatetimeIndex(cutoff_obj, freq='YE-DEC', name=cutoff_obj.name)
                return new_cutoff
            except Exception:
                return cutoff_obj  # Return original if rebuild fails
        
        # Fix main forecaster cutoff
        if hasattr(forecaster, '_cutoff'):
            forecaster._cutoff = fix_single_cutoff(forecaster._cutoff)
        
        # Fix individual forecasters
        fixed_count = 0
        if hasattr(forecaster, 'forecasters_'):
            forecasters_df = forecaster.forecasters_
            
            if isinstance(forecasters_df, pd.DataFrame) and 'forecasters' in forecasters_df.columns:
                for i in range(len(forecasters_df)):
                    individual_f = forecasters_df.iloc[i]['forecasters']
                    
                    if hasattr(individual_f, '_cutoff'):
                        old_cutoff = individual_f._cutoff
                        new_cutoff = fix_single_cutoff(old_cutoff)
                        
                        if new_cutoff is not old_cutoff:  # Check if it changed
                            individual_f._cutoff = new_cutoff
                            fixed_count += 1
                
                print(f"  {model_name}: Fixed {fixed_count} individual forecaster cutoffs")
    
    print("\nApplying cutoff frequency fixes...")
    fix_all_forecaster_cutoffs(hitter_ros, "Hitter ROS")
    fix_all_forecaster_cutoffs(pitcher_ros, "Pitcher ROS")
    print("Cutoff fixes applied.")

    # Load historical data for ROS predictions
    from new_pipeline.notebooks.shared.pipeline_runner import load_historical_data

    print("\nLoading historical data (2016-2024)...")
    hitter_historical_raw = load_historical_data('hitter', range(2016, 2025))
    pitcher_historical_raw = load_historical_data('pitcher', range(2016, 2025))
    print(f"  Loaded {len(hitter_historical_raw)} hitter seasons (raw)")
    print(f"  Loaded {len(pitcher_historical_raw)} pitcher seasons (raw)")

    # Historical data needs same engineered features as current season data
    print("\nProcessing historical data through pipeline...")
    hitter_historical = run_data_pipeline(hitter_historical_raw, player_type='hitter')
    pitcher_historical = run_data_pipeline(pitcher_historical_raw, player_type='pitcher')
    print(f"  Processed {len(hitter_historical)} hitter seasons (qualified)")
    print(f"  Processed {len(pitcher_historical)} pitcher seasons (qualified)")

    # Load historical split data (for ROS model lag features)
    print("\nLoading historical split data (2016-2024)...")
    hitter_splits_path = models_dir / 'hitter_splits_2016_2024.pkl'
    pitcher_splits_path = models_dir / 'pitcher_splits_2016_2024.pkl'

    if hitter_splits_path.exists() and pitcher_splits_path.exists():
        hitter_splits = joblib.load(hitter_splits_path)
        pitcher_splits = joblib.load(pitcher_splits_path)
        print(f"  Loaded hitter splits: {len(hitter_splits)} rows")
        print(f"  Loaded pitcher splits: {len(pitcher_splits)} rows")
        print(f"  Split points available: {sorted(hitter_splits['split_point'].unique())}")
    else:
        print("  Warning: Split data not found. Run ros_training.ipynb first.")
        print(f"    Expected: {hitter_splits_path}")
        print(f"    Expected: {pitcher_splits_path}")
        hitter_splits = None
        pitcher_splits = None

    ros_models_loaded = True
else:
    print("\nROS models not found. Train them first using ros_training.ipynb")
    print(f"  Expected: {hitter_ros_path}")
    print(f"  Expected: {pitcher_ros_path}")
    ros_models_loaded = False

LOADING ROS MODELS FOR COMPLETE PROJECTIONS

Loading trained ROS models...
  Hitter ROS model loaded
  Pitcher ROS model loaded

Loading Darts temporal models...
  Hitter TCN and TSMixer loaded
  Verified: Hitter temporal models have .model attribute
  Pitcher TCN and TSMixer loaded
  Verified: Pitcher temporal models have .model attribute

Applying cutoff frequency fixes...


17:30:46 - new_pipeline.common.transformers.filters - INFO - PAFilter: Removed 1524 hitters with < 75 PA (full season)


  Hitter ROS: Fixed 882 individual forecaster cutoffs
  Pitcher ROS: Fixed 829 individual forecaster cutoffs
Cutoff fixes applied.

Loading historical data (2016-2024)...
  Loaded 5760 hitter seasons (raw)
  Loaded 7237 pitcher seasons (raw)

Processing historical data through pipeline...


17:30:46 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 2088 hitters
17:30:46 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 20-45)
17:30:46 - new_pipeline.common.transformers.hitter_features - INFO - Loading hitter features...
17:30:52 - new_pipeline.common.transformers.hitter_features - INFO - Loaded 11 hitter feature sets (33 total columns)
17:30:52 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 28 features
17:30:52 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 70 missing values
17:30:52 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'K%' range [3.09, 51.25] outside expected [0, 50]
  - Feature 'AVG' range [0.09, 0.38] outside expected [0.1, 0.4]
  - Feature 'OBP' range [0.10, 0.49] outside expected [0.2, 0.5]
  - Feature 'SLG' range [0.13, 0.73] ou

17:30:58 - new_pipeline.common.transformers.pitcher_features - INFO - Loaded 13 pitcher feature sets (38 total columns)
17:30:58 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Calculating pitcher composite features...
17:30:58 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Added 7 composite features
17:30:58 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 41 features
17:30:58 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 10901 missing values
17:30:58 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'BB%' range [0.00, 50.00] outside expected [0, 25]
  - Feature 'K%' range [0.00, 53.00] outside expected [0, 50]
  - Feature 'ERA' range [0.00, 37.50] outside expected [0, 15]
  - Feature 'GB%' range [0.00, 82.79] outside expected [20, 80]
17:30:58 - new_pipeline.common.transformers.feature_s

  Processed 4236 hitter seasons (qualified)
  Processed 5652 pitcher seasons (qualified)

Loading historical split data (2016-2024)...
  Loaded hitter splits: 12702 rows
  Loaded pitcher splits: 10503 rows
  Split points available: [np.float64(0.25), np.float64(0.5), np.float64(0.75)]


In [4]:
# Cell 4: Generate Complete Season Projections (Current + ROS)

if ros_models_loaded:
    print("="*70)
    print("GENERATING COMPLETE SEASON PROJECTIONS")
    print("="*70)
    
    # Create projection generators
    hitter_gen = CompleteProjectionGenerator(
        current_season_model=hitter_model,  # From Cell 2
        ros_model=hitter_ros,
        player_type='hitter'
    )
    
    pitcher_gen = CompleteProjectionGenerator(
        current_season_model=pitcher_model,  # From Cell 2
        ros_model=pitcher_ros,
        player_type='pitcher'
    )
    
    # Generate complete projections
    print("\nGenerating hitter projections...")
    hitter_complete = hitter_gen.generate_projection(
        firsthalf_data=hitter_processed,  # From Cell 2
        historical_data=hitter_historical,  # For feature building
        historical_splits=hitter_splits  # For ROS model lag features
    )
    
    print("\nGenerating pitcher projections...")
    pitcher_complete = pitcher_gen.generate_projection(
        firsthalf_data=pitcher_processed,  # From Cell 2
        historical_data=pitcher_historical,  # For feature building
        historical_splits=pitcher_splits,  # For ROS model lag features
        team_games_source_df=hitter_processed  #Use hitter data for team games
    )
    
    print(f"\nComplete! Generated projections for:")
    print(f"  {len(hitter_complete)} hitters")
    print(f"  {len(pitcher_complete)} pitchers")
else:
    print("Skipping complete projections (ROS models not loaded)")

GENERATING COMPLETE SEASON PROJECTIONS

Generating hitter projections...
Generating complete projections for 488 hitters...
  Step 1: Running current season model...
  Step 2: Calculating actual firsthalf WAR...
  Step 3: Building ROS features...
    Built features for 488 players
    Checking for missing features...
      Primary_Position: 84/488 players missing
      _multi_team_current: 472/488 players missing
      _multi_team_stints: 472/488 players missing
    Example player with NaN: Aaron Judge
      Missing features (2 total): ['_multi_team_current', '_multi_team_stints']
  Step 4: Running ROS model...
    Filtering historical splits to nearest match (current: 0.50)...
    Using split_point=0.5 (4234 historical observations)
  Player tiers: Tier1=221, Tier2=56, Tier3=211


c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\sktime\datatypes\_vectorize.py:404: FutureWarning: The behavior of pd.concat with len(keys) != len(objs) is deprecated. In a future version this will raise instead of truncating to the smaller of the two sequences
  X_mi_reconstructed = pd.concat(df_list, keys=row_ix, axis=0)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: T

  Step 4.5: Applying rookie/call-up regression...
    Applied regression to 318 players below qualification standards
  Step 5: Projecting remaining usage...
  Step 6: Using ROS WAR predictions...
  Step 7: Combining projections...
Complete! Generated projections for 488 hitters

Generating pitcher projections...
Generating complete projections for 597 pitchers...
  Step 1: Running current season model...
  Step 2: Calculating actual firsthalf WAR...
  Step 3: Building ROS features...
    Built features for 597 players
    Checking for missing features...
      _multi_team_current: 568/597 players missing
      _multi_team_stints: 568/597 players missing
    Example player with NaN: Tarik Skubal
      Missing features (2 total): ['_multi_team_current', '_multi_team_stints']
  Step 4: Running ROS model...
    Filtering historical splits to nearest match (current: 0.50)...
    Using split_point=0.5 (3501 historical observations)
  Player tiers: Tier1=189, Tier2=52, Tier3=356


c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\sktime\datatypes\_vectorize.py:404: FutureWarning: The behavior of pd.concat with len(keys) != len(objs) is deprecated. In a future version this will raise instead of truncating to the smaller of the two sequences
  X_mi_reconstructed = pd.concat(df_list, keys=row_ix, axis=0)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: T

  Step 4.5: Applying rookie/call-up regression...
    Applied regression to 350 players below qualification standards
  Step 5: Projecting remaining usage...
  Step 6: Using ROS WAR predictions...
  Step 7: Combining projections...
Complete! Generated projections for 597 pitchers

Complete! Generated projections for:
  488 hitters
  597 pitchers


In [5]:
# Cell 5: Pitcher Projections Scatter

if ros_models_loaded:
    # Add pitcher type for coloring
    pitcher_complete_copy = pitcher_complete.copy()
    pitcher_complete_copy['Type'] = pitcher_complete_copy.apply(
        lambda row: 'Starter' if row.get('GS_per_G', row.get('GS', 0) / max(row.get('G', 1), 1)) > 0.7
        else ('Reliever' if row.get('GS_per_G', row.get('GS', 0) / max(row.get('G', 1), 1)) < 0.1
              else 'Swing'),
        axis=1
    )

    fig_pitchers = create_war_scatter(
        df=pitcher_complete_copy,
        player_type='pitcher',
        title="2025 Pitcher WAR Projections",
        color_by='Type',
        hover_data=['Name', 'Team', 'Type', 'Total_Projected_IP', 'Current_WAR', 'ROS_WAR', 'Total_Projected_WAR']
    )

    fig_pitchers.show()
else:
    print("Skipping pitcher scatter (ROS models not loaded)")

In [7]:
# Cell 6: Hitter Projections Scatter

if ros_models_loaded:
    fig_hitters = create_war_scatter(
        df=hitter_complete,
        player_type='hitter',
        title="2025 Hitter WAR Projections",
        color_by='Primary_Position',
        hover_data=['Name', 'Team', 'Primary_Position', 'Total_Projected_PA', 'Current_WAR', 'ROS_WAR', 'Total_Projected_WAR']
    )

    fig_hitters.show()
else:
    print("Skipping hitter scatter (ROS models not loaded)")

In [8]:
# Cell 7: Featured Pitchers Table

if ros_models_loaded:
    # Define featured pitchers (manually curated list)
    featured_pitchers = [
        'Tarik Skubal',
        'Paul Skenes',
        'Yoshinobu Yamamoto',
        'Aroldis Chapman',
        'Garrett Crochet',
        'Zack Wheeler',
        'Corbin Burnes',
        'Chris Sale',
        'Emmanuel Clase'
    ]

    # Check if Shohei Ohtani is in both datasets (two-way player)
    two_way_data = None
    if 'Shohei Ohtani' in pitcher_complete['Name'].values and 'Shohei Ohtani' in hitter_complete['Name'].values:
        featured_pitchers.append('Shohei Ohtani')
        
        ohtani_pitcher = pitcher_complete[pitcher_complete['Name'] == 'Shohei Ohtani'].iloc[0]
        ohtani_hitter = hitter_complete[hitter_complete['Name'] == 'Shohei Ohtani'].iloc[0]
        
        two_way_data = {
            'Shohei Ohtani': handle_two_way_player(
                pitcher_war={'current': ohtani_pitcher['Current_WAR'],
                            'ROS': ohtani_pitcher['ROS_WAR'],
                            'total': ohtani_pitcher['Total_Projected_WAR']},
                hitter_war={'current': ohtani_hitter['Current_WAR'],
                           'ROS': ohtani_hitter['ROS_WAR'],
                           'total': ohtani_hitter['Total_Projected_WAR']}
            )
        }

    # Create table
    pitcher_table = create_featured_table(
        df=pitcher_complete,
        player_names=featured_pitchers,
        player_type='pitcher',
        two_way_data=two_way_data
    )

    print("=" * 99)
    print("FEATURED PITCHERS")
    print("=" * 99)
    print(pitcher_table)
else:
    print("Skipping featured pitchers table (ROS models not loaded)")

FEATURED PITCHERS
| Type Rank | Overall | Name               | Type     | Team | Current WAR | ROS WAR | Total Proj |
|---------: |-------: | :------------------| :--------| :----|-----------: |-------: |----------: |
|     1/198 |   1/597 | Tarik Skubal       | Starter  | DET  |         4.7 |     2.5 |        7.2 |
|     3/198 |   3/597 | Paul Skenes        | Starter  | PIT  |         3.8 |     2.3 |        6.0 |
|     9/198 |   9/597 | Yoshinobu Yamamoto | Starter  | LAD  |         2.8 |     1.9 |        4.8 |
|     1/338 |  46/597 | Aroldis Chapman    | Reliever | BOS  |         1.7 |     0.8 |        2.5 |
|     2/198 |   2/597 | Garrett Crochet    | Starter  | BOS  |         4.2 |     2.2 |        6.4 |
|     4/198 |   4/597 | Zack Wheeler       | Starter  | PHI  |         4.0 |     1.9 |        5.9 |
|    74/198 |  78/597 | Corbin Burnes      | Starter  | ARI  |         1.4 |     0.5 |        1.9 |
|    19/198 |  19/597 | Chris Sale         | Starter  | ATL  |         2.5 |     1

In [9]:
# Cell 8: Featured Hitters Table

if ros_models_loaded:
    # Define featured hitters (manually curated list)
    featured_hitters = [
        'Aaron Judge',
        'Juan Soto',
        'Bobby Witt Jr.',
        'Freddie Freeman',
        'Mookie Betts',
        'Cal Raleigh',
        'Kyle Schwarber'
    ]

    if 'Shohei Ohtani' in hitter_complete['Name'].values:
        featured_hitters.append('Shohei Ohtani')

    # Create table (two_way_data already defined from Cell 7)
    hitter_table = create_featured_table(
        df=hitter_complete,
        player_names=featured_hitters,
        player_type='hitter',
        two_way_data=two_way_data
    )

    print("=" * 94)
    print("FEATURED HITTERS")
    print("=" * 94)
    print(hitter_table)
else:
    print("Skipping featured hitters table (ROS models not loaded)")

FEATURED HITTERS
| Pos Rank | Overall | Name            | Pos     | Team | Current WAR | ROS WAR | Total Proj |
|--------: |-------: | :---------------| :-------| :----|-----------: |-------: |----------: |
|     1/94 |   1/488 | Aaron Judge     | RF      | NYY  |         6.3 |     3.8 |       10.1 |
|     3/94 |   7/488 | Juan Soto       | RF      | NYM  |         4.1 |     3.2 |        7.3 |
|     1/63 |   6/488 | Bobby Witt Jr.  | SS      | KCR  |         4.4 |     2.9 |        7.4 |
|     5/36 |  57/488 | Freddie Freeman | 1B      | LAD  |         1.9 |     1.9 |        3.8 |
|    20/63 |  80/488 | Mookie Betts    | SS      | LAD  |         1.7 |     1.5 |        3.2 |
|     1/63 |   2/488 | Cal Raleigh     | C       | SEA  |         6.7 |     3.1 |        9.8 |
|     2/50 |  16/488 | Kyle Schwarber  | LF      | PHI  |         3.7 |     2.6 |        6.3 |
|      N/A |   3/488 | Shohei Ohtani   | Two-Way | LAD  |         5.3 |     4.2 |        9.5 |


In [10]:
# Cell 9: Top Projected Hitters (Complete Season)

if ros_models_loaded:
    print("="*99)
    print("TOP 10 PROJECTED WAR - HITTERS (2025 Complete Season)")
    print("="*99)
    print()
    
    top_hitters = hitter_complete.nlargest(10, 'Total_Projected_WAR')
    display_cols = [
        'Name', 'Team', 'Current_WAR', 'ROS_WAR',
        'Total_Projected_WAR', 'Q10_WAR', 'Q90_WAR', 'Uncertainty_Range'
    ]
    
    # Round for display
    top_display = top_hitters[display_cols].copy()
    for col in ['Current_WAR', 'ROS_WAR', 'Total_Projected_WAR', 'Q10_WAR', 'Q90_WAR', 'Uncertainty_Range']:
        top_display[col] = top_display[col].round(1)
    
    print(top_display.to_string(index=False))
    print()
    print("="*99)
else:
    print("Skipping (ROS models not loaded)")

TOP 10 PROJECTED WAR - HITTERS (2025 Complete Season)

               Name Team  Current_WAR  ROS_WAR  Total_Projected_WAR  Q10_WAR  Q90_WAR  Uncertainty_Range
        Aaron Judge  NYY          6.3      3.8                 10.1      7.4     11.3                3.9
        Cal Raleigh  SEA          6.7      3.1                  9.8      7.7     10.8                3.1
      Shohei Ohtani  LAD          5.0      3.3                  8.3      6.1      8.8                2.7
        Kyle Tucker  CHC          4.5      3.2                  7.6      5.5      8.2                2.7
       José Ramírez  CLE          4.5      3.0                  7.5      5.6      8.3                2.8
     Bobby Witt Jr.  KCR          4.4      2.9                  7.4      5.5      8.2                2.8
          Juan Soto  NYM          4.1      3.2                  7.3      5.2      7.6                2.5
       Byron Buxton  MIN          4.4      2.9                  7.3      5.4      8.3                2.9


In [11]:
# Cell 10: Top Projected Pitchers (Complete Season)

if ros_models_loaded:
    print("="*70)
    print("TOP 10 PROJECTED WAR - PITCHERS (2025 Complete Season)")
    print("="*70)
    print()
    
    top_pitchers = pitcher_complete.nlargest(10, 'Total_Projected_WAR')
    display_cols = [
        'Name', 'Team', 'Current_WAR', 'ROS_WAR',
        'Total_Projected_WAR', 'Q10_WAR', 'Q90_WAR', 'Uncertainty_Range'
    ]
    
    # Round for display
    top_display = top_pitchers[display_cols].copy()
    for col in ['Current_WAR', 'ROS_WAR', 'Total_Projected_WAR', 'Q10_WAR', 'Q90_WAR', 'Uncertainty_Range']:
        top_display[col] = top_display[col].round(1)
    
    print(top_display.to_string(index=False))
    print()
    print("="*70)
else:
    print("Skipping (ROS models not loaded)")

TOP 10 PROJECTED WAR - PITCHERS (2025 Complete Season)

              Name Team  Current_WAR  ROS_WAR  Total_Projected_WAR  Q10_WAR  Q90_WAR  Uncertainty_Range
      Tarik Skubal  DET          4.7      2.5                  7.2      5.7      7.5                1.7
   Garrett Crochet  BOS          4.2      2.2                  6.4      5.2      6.7                1.4
       Paul Skenes  PIT          3.8      2.3                  6.0      4.8      6.3                1.5
      Zack Wheeler  PHI          4.0      1.9                  5.9      5.0      6.0                1.0
Cristopher Sánchez  PHI          3.3      1.9                  5.1      4.3      5.3                1.0
        Logan Webb  SFG          3.5      1.7                  5.1      4.4      5.3                0.9
      Hunter Brown  HOU          3.4      1.6                  5.0      4.5      5.3                0.9
    Nathan Eovaldi  TEX          3.0      2.0                  5.0      4.1      5.0                1.0
Yoshinob